In [ ]:
# E23 P2: Merlin 3.0.0 offline payload + offline install test on the pinned image (CPU, internet ON)
import json, os, re, shutil, subprocess, sys, time, hashlib
from pathlib import Path
T0 = time.time()
FILES = json.loads('{"glibc235_fix.py": "\\"\\"\\"E23 P2: make the Merlin release .so files load on glibc 2.35 (Kaggle image, Ubuntu 22.04).\\n\\nThe Merlin 3.0.0 wheel\'s vllm/_C_stable_libtorch.abi3.so needs libc.so.6 version GLIBC_2.38, only for\\n__isoc23_strtol / __isoc23_strtoll (C23 names of strtol / strtoll that GCC 13 + glibc 2.38 headers emit).\\nThis tool, per ELF file:\\n  1. lists every dynamic symbol bound to a libc.so.6 version above MAX_OK (2.35);\\n  2. refuses (exit 2) if any such symbol is not one the shim provides (SHIM_SYMBOLS);\\n  3. sets VER_FLG_WEAK on those version-need entries, so ld.so does not fail on the missing version\\n     (glibc dl-version.c: a weak version that is not found is not fatal); the symbols then bind to the\\n     unversioned definitions in the LD_PRELOADed shim (dl-lookup.c accepts an unversioned definition\\n     from an object that has no version info).\\nNo other byte changes. Standard library only.\\nusage: python glibc235_fix.py [--dry-run] <file.so> ...   -> prints one GLIBCFIX {...} JSON line\\"\\"\\"\\nimport json\\nimport struct\\nimport sys\\n\\nMAX_OK = (2, 35)\\nSHIM_SYMBOLS = {\\"__isoc23_strtol\\", \\"__isoc23_strtoll\\", \\"__isoc23_strtoul\\", \\"__isoc23_strtoull\\"}\\nSHT_DYNSYM, SHT_GNU_VERNEED, SHT_GNU_VERSYM = 11, 0x6FFFFFFE, 0x6FFFFFFF\\nVER_FLG_WEAK = 0x2\\n\\n\\ndef glibc_ver(name):\\n    if not name.startswith(\\"GLIBC_\\"):\\n        return None\\n    try:\\n        return tuple(int(x) for x in name[6:].split(\\".\\"))\\n    except ValueError:\\n        return None\\n\\n\\ndef cstr(buf, off):\\n    return buf[off:buf.index(b\\"\\\\0\\", off)].decode()\\n\\n\\ndef scan(path):\\n    data = bytearray(open(path, \\"rb\\").read())\\n    if data[:4] != b\\"\\\\x7fELF\\" or data[4] != 2 or data[5] != 1:\\n        raise ValueError(\\"not a little-endian ELF64 file\\")\\n    e_shoff, = struct.unpack_from(\\"<Q\\", data, 0x28)\\n    e_shentsize, e_shnum = struct.unpack_from(\\"<HH\\", data, 0x3A)\\n    secs = []\\n    for i in range(e_shnum):\\n        sh = struct.unpack_from(\\"<IIQQQQIIQQ\\", data, e_shoff + i * e_shentsize)\\n        secs.append({\\"type\\": sh[1], \\"offset\\": sh[4], \\"size\\": sh[5], \\"link\\": sh[6], \\"entsize\\": sh[9]})\\n    by_type = {}\\n    for s in secs:\\n        by_type.setdefault(s[\\"type\\"], s)\\n    vn = by_type.get(SHT_GNU_VERNEED)\\n    rows = {\\"needs\\": [], \\"high\\": [], \\"high_symbols\\": []}\\n    if vn is None:\\n        return data, rows\\n    strtab = secs[vn[\\"link\\"]]\\n    st = bytes(data[strtab[\\"offset\\"]:strtab[\\"offset\\"] + strtab[\\"size\\"]])\\n    off, high_idx = vn[\\"offset\\"], {}\\n    while True:\\n        vn_version, vn_cnt, vn_file, vn_aux, vn_next = struct.unpack_from(\\"<HHIII\\", data, off)\\n        lib = cstr(st, vn_file)\\n        aoff = off + vn_aux\\n        for _ in range(vn_cnt):\\n            vna_hash, vna_flags, vna_other, vna_name, vna_next = struct.unpack_from(\\"<IHHII\\", data, aoff)\\n            name = cstr(st, vna_name)\\n            rows[\\"needs\\"].append([lib, name, vna_flags, vna_other])\\n            v = glibc_ver(name)\\n            if lib == \\"libc.so.6\\" and v and v > MAX_OK:\\n                rows[\\"high\\"].append({\\"lib\\": lib, \\"version\\": name, \\"flags\\": vna_flags, \\"index\\": vna_other,\\n                                     \\"flags_offset\\": aoff + 4})\\n                high_idx[vna_other] = name\\n            if not vna_next:\\n                break\\n            aoff += vna_next\\n        if not vn_next:\\n            break\\n        off += vn_next\\n    if high_idx:\\n        dynsym, versym = by_type.get(SHT_DYNSYM), by_type.get(SHT_GNU_VERSYM)\\n        dstr = secs[dynsym[\\"link\\"]]\\n        ds = bytes(data[dstr[\\"offset\\"]:dstr[\\"offset\\"] + dstr[\\"size\\"]])\\n        n = dynsym[\\"size\\"] // dynsym[\\"entsize\\"]\\n        for i in range(n):\\n            st_name, = struct.unpack_from(\\"<I\\", data, dynsym[\\"offset\\"] + i * dynsym[\\"entsize\\"])\\n            ndx, = struct.unpack_from(\\"<H\\", data, versym[\\"offset\\"] + 2 * i)\\n            if (ndx & 0x7FFF) in high_idx:\\n                rows[\\"high_symbols\\"].append([cstr(ds, st_name), high_idx[ndx & 0x7FFF]])\\n    return data, rows\\n\\n\\ndef main(argv):\\n    dry = \\"--dry-run\\" in argv\\n    files = [a for a in argv if a != \\"--dry-run\\"]\\n    out = {\\"dry_run\\": dry, \\"files\\": {}, \\"ok\\": True}\\n    for f in files:\\n        rec = {}\\n        try:\\n            data, rows = scan(f)\\n            rec.update(high=[{k: v for k, v in h.items() if k != \\"flags_offset\\"} for h in rows[\\"high\\"]],\\n                       high_symbols=rows[\\"high_symbols\\"])\\n            unknown = sorted({s for s, _ in rows[\\"high_symbols\\"]} - SHIM_SYMBOLS)\\n            rec[\\"unknown_symbols\\"] = unknown\\n            if unknown:\\n                rec[\\"action\\"] = \\"REFUSED: symbols above glibc 2.35 that the shim does not provide\\"\\n                out[\\"ok\\"] = False\\n            elif not rows[\\"high\\"]:\\n                rec[\\"action\\"] = \\"none needed\\"\\n            else:\\n                for h in rows[\\"high\\"]:\\n                    fl, = struct.unpack_from(\\"<H\\", data, h[\\"flags_offset\\"])\\n                    struct.pack_into(\\"<H\\", data, h[\\"flags_offset\\"], fl | VER_FLG_WEAK)\\n                if not dry:\\n                    open(f, \\"wb\\").write(data)\\n                    _, again = scan(f)\\n                    rec[\\"after\\"] = [[h[\\"version\\"], h[\\"flags\\"]] for h in again[\\"high\\"]]\\n                    if not all(h[\\"flags\\"] & VER_FLG_WEAK for h in again[\\"high\\"]):\\n                        rec[\\"action\\"] = \\"FAILED: weak flag not set after write\\"\\n                        out[\\"ok\\"] = False\\n                        out[\\"files\\"][f] = rec\\n                        continue\\n                rec[\\"action\\"] = \\"weak flag set\\" + (\\" (dry run, not written)\\" if dry else \\"\\")\\n        except Exception as e:\\n            rec[\\"action\\"] = f\\"ERROR {type(e).__name__}: {e}\\"\\n            out[\\"ok\\"] = False\\n        out[\\"files\\"][f] = rec\\n    print(\\"GLIBCFIX \\" + json.dumps(out))\\n    return 0 if out[\\"ok\\"] else 2\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main(sys.argv[1:]))\\n", "rehash_products.py": "\\"\\"\\"E23 P2: after glibc235_fix.py changed .so bytes, re-record <tree>/.build-products.sha256 for the SAME file list\\n(the format Merlin\'s install-env.sh writes: \'<sha256>  <path relative to tree>\'), so launch.py\'s product check and\\ninstall-env.sh --verify accept the patched files. Every other line keeps its hash; a changed line is reported.\\nusage: python rehash_products.py <merlin-tree>   -> prints one REHASH {...} JSON line\\"\\"\\"\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\ntree = Path(sys.argv[1])\\nsums = tree / \\".build-products.sha256\\"\\nlines = sums.read_text(encoding=\\"utf-8\\").splitlines()\\nout, changed = [], []\\nfor ln in lines:\\n    digest, rel = ln.split(\\"  \\", 1)\\n    new = hashlib.sha256((tree / rel).read_bytes()).hexdigest()\\n    if new != digest:\\n        changed.append({\\"path\\": rel, \\"old\\": digest, \\"new\\": new})\\n    out.append(f\\"{new}  {rel}\\\\n\\")\\nsums.write_text(\\"\\".join(out), encoding=\\"utf-8\\")\\nprint(\\"REHASH \\" + json.dumps({\\"files\\": len(lines), \\"changed\\": changed}))\\n", "glibc238_shim.c": "/* E23 P2: the C23 integer-parsing entry points glibc 2.38 added (__isoc23_strto*), for glibc 2.35.\\n   The Merlin release .so files call them only through GCC 13 + glibc 2.38 headers; C23 differs from C11 only in\\n   accepting a \\"0b\\"/\\"0B\\" prefix with base 0 or 2, which these wrappers add. Built without a version script, so its\\n   symbols are unversioned (they satisfy the weakened GLIBC_2.38 references, see glibc235_fix.py). LD_PRELOAD it. */\\n#include <stdlib.h>\\n#include <ctype.h>\\n\\nstatic const char *bin_body(const char *s, int base, int *neg, const char **start)\\n{\\n    const char *p = s;\\n    while (isspace((unsigned char)*p))\\n        p++;\\n    *start = p;\\n    *neg = 0;\\n    if (*p == \'+\' || *p == \'-\') {\\n        *neg = (*p == \'-\');\\n        p++;\\n    }\\n    if ((base == 0 || base == 2) && p[0] == \'0\' && (p[1] == \'b\' || p[1] == \'B\') && (p[2] == \'0\' || p[2] == \'1\'))\\n        return p + 2;\\n    return NULL;\\n}\\n\\n#define DEFINE(NAME, T, F)                                                                 \\\\\\n    T NAME(const char *s, char **end, int base)                                            \\\\\\n    {                                                                                      \\\\\\n        int neg;                                                                           \\\\\\n        const char *start, *b = bin_body(s, base, &neg, &start);                           \\\\\\n        if (b) {                                                                           \\\\\\n            T v = (T)F(b, end, 2);                                                         \\\\\\n            return neg ? (T)(0 - v) : v;                                                   \\\\\\n        }                                                                                  \\\\\\n        return F(s, end, base);                                                            \\\\\\n    }\\n\\nDEFINE(__isoc23_strtol, long, strtol)\\nDEFINE(__isoc23_strtoll, long long, strtoll)\\nDEFINE(__isoc23_strtoul, unsigned long, strtoul)\\nDEFINE(__isoc23_strtoull, unsigned long long, strtoull)\\n", "install_merlin_offline.sh": "#!/usr/bin/env bash\\n# E23 P2: install the Merlin 3.0.0 engine (vLLM 0.32.0 fork) OFFLINE on the pinned Kaggle image (Ubuntu 22.04,\\n# glibc 2.35, Python 3.12, GCC 11) from the P2 payload. Runs Merlin\'s own release/install-env.sh unchanged:\\n#   - CPython 3.13 from the payload (uv-managed install, fixed path /tmp/merlin_uvpy), no Python download;\\n#   - every locked dependency from the payload wheelhouse (uv.toml: no-index + find-links), UV_OFFLINE=1;\\n#   - GCC 13 from conda-forge (payload, fixed path /tmp/merlin_gcc13) for the installer\'s GCC >= 13 rule;\\n#   - the release wheel through MERLIN_WHEEL (precompiled products, no native build).\\n# Then the glibc 2.35 fix (glibc235_fix.py: weak GLIBC_2.38 need + LD_PRELOAD shim), the build-product hashes\\n# re-recorded for the patched files, and install-env.sh --verify.\\n#   bash install_merlin_offline.sh <payload-dir> <dest-dir>\\n# Prints MERLIN_VENV= / MERLIN_TREE= / MERLIN_SHIM= / MERLIN_CC= / MERLIN_CXX= lines and INSTALL OK at the end.\\nset -euo pipefail\\nPAY=\\"$(cd \\"${1:?usage: install_merlin_offline.sh <payload-dir> <dest-dir>}\\" && pwd -P)\\"\\nDEST=\\"${2:?usage: install_merlin_offline.sh <payload-dir> <dest-dir>}\\"\\nmkdir -p \\"$DEST\\"\\nDEST=\\"$(cd \\"$DEST\\" && pwd -P)\\"\\ndie() { echo \\"install_merlin_offline: FAIL: $*\\" >&2; exit 1; }\\n# Kaggle Models refuse \'+\' in file names, so the payload stores the wheel as ..._PLUS_merlin...; Merlin\'s installer\\n# verifies the wheel\'s basename against release/merlin-wheel.sha256, so it is copied back to its real name here.\\nWHEEL_SAFE=\\"$(ls \\"$PAY\\"/vllm-0.32.0_PLUS_merlin.*-cp313-cp313-linux_x86_64.whl 2>/dev/null | head -1)\\"\\nif [ -f \\"$WHEEL_SAFE\\" ]; then\\n  mkdir -p \\"$DEST/wheel\\"\\n  WHEEL=\\"$DEST/wheel/$(basename \\"$WHEEL_SAFE\\" | sed \'s/_PLUS_/+/\')\\"\\n  cp -f \\"$WHEEL_SAFE\\" \\"$WHEEL\\"\\nelse\\n  WHEEL=\\"$(ls \\"$PAY\\"/vllm-0.32.0+merlin.*-cp313-cp313-linux_x86_64.whl 2>/dev/null | head -1)\\"\\nfi\\n[ -f \\"$WHEEL\\" ] || die \\"no Merlin wheel in $PAY\\"\\nfor f in merlin-engine.tar merlin_uvpy.tar.gz merlin_gcc13.tar.gz glibc235_fix.py rehash_products.py libglibc238_shim.so; do\\n  [ -e \\"$PAY/$f\\" ] || die \\"payload is missing $f\\"\\ndone\\n[ -d \\"$PAY/wheelhouse\\" ] || die \\"payload is missing wheelhouse/\\"\\n\\n# fixed-prefix trees (the conda env and the uv Python record their own install prefix)\\n[ -x /tmp/merlin_uvpy/.ok ] || { rm -rf /tmp/merlin_uvpy; tar -xzf \\"$PAY/merlin_uvpy.tar.gz\\" -C /tmp; touch /tmp/merlin_uvpy/.ok; chmod +x /tmp/merlin_uvpy/.ok; }\\n[ -x /tmp/merlin_gcc13/.ok ] || { rm -rf /tmp/merlin_gcc13; tar -xzf \\"$PAY/merlin_gcc13.tar.gz\\" -C /tmp; touch /tmp/merlin_gcc13/.ok; chmod +x /tmp/merlin_gcc13/.ok; }\\nCC13=\\"$(ls /tmp/merlin_gcc13/bin/x86_64-conda-linux-gnu-gcc 2>/dev/null || true)\\"\\nCXX13=\\"$(ls /tmp/merlin_gcc13/bin/x86_64-conda-linux-gnu-g++ 2>/dev/null || true)\\"\\n[ -x \\"$CC13\\" ] && [ -x \\"$CXX13\\" ] || die \\"GCC 13 not found under /tmp/merlin_gcc13/bin\\"\\n\\nTREE=\\"$DEST/merlin-engine\\"\\nVENV=\\"$DEST/venv\\"\\n[ ! -e \\"$VENV\\" ] || die \\"$VENV already exists\\"\\nrm -rf \\"$TREE\\"\\ntar -xf \\"$PAY/merlin-engine.tar\\" -C \\"$DEST\\"\\n[ -f \\"$TREE/release/install-env.sh\\" ] || die \\"merlin-engine.tar did not give $TREE\\"\\ngit config --global --add safe.directory \\"$TREE\\" 2>/dev/null || true\\n\\ncat > \\"$DEST/uv-offline.toml\\" <<EOF\\n[pip]\\nno-index = true\\nfind-links = [\\"$PAY/wheelhouse\\"]\\nEOF\\nexport UV_CONFIG_FILE=\\"$DEST/uv-offline.toml\\" UV_OFFLINE=1 UV_PYTHON_DOWNLOADS=never\\nexport UV_PYTHON_INSTALL_DIR=/tmp/merlin_uvpy UV_CACHE_DIR=\\"$DEST/uv-cache\\" UV_LINK_MODE=copy\\nexport CC=\\"$CC13\\" CXX=\\"$CXX13\\" CUDAHOSTCXX=\\"$CXX13\\"\\nunset CUDA_HOME CUDA_PATH PYTHONPATH PYTHONHOME VLLM_USE_PRECOMPILED VLLM_PRECOMPILED_WHEEL_LOCATION CMAKE_ARGS || true\\nexport MERLIN_WHEEL=\\"$WHEEL\\"\\n(cd \\"$TREE\\" && bash release/install-env.sh \\"$VENV\\")\\nunset MERLIN_WHEEL\\n\\nmapfile -t SOS < <(find \\"$TREE/vllm\\" -type f \\\\( -name \'*.so\' -o -name \'*.so.*\' \\\\))\\npython3 -I \\"$PAY/glibc235_fix.py\\" \\"${SOS[@]}\\"\\npython3 -I \\"$PAY/rehash_products.py\\" \\"$TREE\\"\\ncp -f \\"$PAY/libglibc238_shim.so\\" \\"$DEST/libglibc238_shim.so\\"\\n(cd \\"$TREE\\" && bash release/install-env.sh --verify \\"$VENV\\")\\n\\necho \\"MERLIN_VENV=$VENV\\"\\necho \\"MERLIN_TREE=$TREE\\"\\necho \\"MERLIN_SHIM=$DEST/libglibc238_shim.so\\"\\necho \\"MERLIN_CC=$CC13\\"\\necho \\"MERLIN_CXX=$CXX13\\"\\necho \\"INSTALL OK\\"\\n"}')
TAG, COMMIT = "merlin-v3.0.0", "ad42e37feb7611c4ffd24017bf20a7e2dc94c6e5"
WHEEL_NAME = "vllm-0.32.0+merlin.7002f56537-cp313-cp313-linux_x86_64.whl"
WHEEL_SAFE = WHEEL_NAME.replace("+", "_PLUS_")
WHEEL_SHA = "5c33b3dce51b8255b0c1a95a930edb24c1a481a023327d8d2d8f76e8be45c82d"
WHEEL_URL = f"https://github.com/halt95/merlin-engine/releases/download/{TAG}/{WHEEL_NAME}"
W = Path("/kaggle/working"); PAY = W / "merlin_offline"; T = Path("/tmp/e23p2"); LOGS = T / "logs"
for p in (PAY, T, LOGS):
    p.mkdir(parents=True, exist_ok=True)
R = {"steps": {}, "errors": []}
os.environ.pop("CUDA_HOME", None)


def sh(name, cmd, timeout=3600, env=None, ok_rc=(0,)):
    t = time.time()
    log = LOGS / f"{name}.log"
    with open(log, "w") as f:
        r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,
                           env={**os.environ, **(env or {})})
    txt = log.read_text(errors="replace")
    rec = {"rc": r.returncode, "sec": round(time.time() - t, 1), "tail": txt[-2500:]}
    R["steps"][name] = rec
    print(f"[{time.time() - T0:7.1f}s] {name}: rc {r.returncode} ({rec['sec']} s)", flush=True)
    if r.returncode not in ok_rc:
        print(txt[-2500:], flush=True)
    return r.returncode, txt


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(8 << 20), b""):
            h.update(b)
    return h.hexdigest()


def fail(msg):
    R["errors"].append(msg)
    print("ERROR:", msg, flush=True)


try:
    sh("env", "gcc --version | head -1; ldd --version | head -1; uv --version; nproc; free -g | head -2; "
              "df -h /kaggle/working /tmp | cat; python3 --version; git --version")
    for n, s in FILES.items():
        (PAY / n).write_text(s)
    os.chmod(PAY / "install_merlin_offline.sh", 0o755)

    # 1. engine source at the release tag, with .git (install-env.sh checks the checkout's commit / tree)
    shutil.rmtree(T / "src", ignore_errors=True)
    rc, _ = sh("clone", f"git clone --quiet --depth 1 --branch {TAG} https://github.com/halt95/merlin-engine.git {T}/src/merlin-engine "
                        f"&& git -C {T}/src/merlin-engine rev-parse HEAD", timeout=900)
    head = (R["steps"]["clone"]["tail"].strip().splitlines() or [""])[-1]
    R["merlin_commit"] = head
    if rc or head != COMMIT:
        fail(f"merlin checkout: rc {rc}, HEAD {head} != {COMMIT}")
    sh("engine_tar", f"tar -cf {PAY}/merlin-engine.tar -C {T}/src merlin-engine")
    tag_sum = (T / "src/merlin-engine/release/merlin-wheel.sha256").read_text().split()
    R["tag_wheel_sha256"] = tag_sum
    if tag_sum[:2] != [WHEEL_SHA, WHEEL_NAME]:
        fail(f"release/merlin-wheel.sha256 at the tag is {tag_sum}, expected {WHEEL_SHA} {WHEEL_NAME}")

    # 2. the release wheel
    rc, _ = sh("wheel", f"curl -fsSL --retry 5 -o {PAY}/{WHEEL_NAME} {WHEEL_URL} && ls -la {PAY}/{WHEEL_NAME}", timeout=1800)
    got = sha256(PAY / WHEEL_NAME) if (PAY / WHEEL_NAME).is_file() else None
    R["wheel"] = {"sha256": got, "ok": got == WHEEL_SHA}
    if got != WHEEL_SHA:
        fail(f"wheel sha256 {got} != {WHEEL_SHA}")
    # v5: Kaggle Models refuse '+' in file names (v4 Model creation failed); the payload keeps the wheel as WHEEL_SAFE and
    # install_merlin_offline.sh copies it back to WHEEL_NAME before Merlin's installer checks the name
    if (PAY / WHEEL_NAME).is_file():
        os.replace(PAY / WHEEL_NAME, PAY / WHEEL_SAFE)

    # 3. GCC 13 (conda-forge) at the fixed prefix /tmp/merlin_gcc13: install-env.sh requires CC/CXX >= 13
    shutil.rmtree("/tmp/merlin_gcc13", ignore_errors=True)
    # /usr/local/bin/mamba on this image is NOT conda's mamba (v1 run: a Python test runner's CLI); use micromamba's static
    # release binary (sha256-checked against its own .sha256 asset)
    MM = "https://github.com/mamba-org/micromamba-releases/releases/download/2.10.0-0/micromamba-linux-64"
    rc, _ = sh("gcc13", f"curl -fsSL --retry 5 -o /tmp/micromamba {MM} && curl -fsSL --retry 5 -o /tmp/micromamba.sha256 {MM}.sha256 && "
                        "echo \"$(head -c 64 /tmp/micromamba.sha256)  /tmp/micromamba\" | sha256sum -c - && chmod +x /tmp/micromamba && "
                        "/tmp/micromamba --version && export MAMBA_ROOT_PREFIX=/tmp/mambaroot && "
                        "/tmp/micromamba create -y -q -p /tmp/merlin_gcc13 -c conda-forge "
                        "--override-channels 'gcc_linux-64=13' 'gxx_linux-64=13' && "
                        "/tmp/merlin_gcc13/bin/x86_64-conda-linux-gnu-gcc -dumpversion && "
                        "/tmp/merlin_gcc13/bin/x86_64-conda-linux-gnu-g++ -dumpversion && "
                        "echo 'int main(){return 0;}' > /tmp/t.cc && /tmp/merlin_gcc13/bin/x86_64-conda-linux-gnu-g++ -std=c++20 /tmp/t.cc -o /tmp/t && /tmp/t && echo CXX20 OK",
                        timeout=2400)
    R["gcc13"] = {"ok": rc == 0 and "CXX20 OK" in R["steps"]["gcc13"]["tail"]}
    if not R["gcc13"]["ok"]:
        fail("GCC 13 from conda-forge failed")
    sh("gcc13_tar", f"tar -czf {PAY}/merlin_gcc13.tar.gz -C /tmp merlin_gcc13 && du -sh /tmp/merlin_gcc13 {PAY}/merlin_gcc13.tar.gz")

    # 4. CPython 3.13 (uv-managed) at the fixed prefix /tmp/merlin_uvpy
    shutil.rmtree("/tmp/merlin_uvpy", ignore_errors=True)
    rc, _ = sh("py313", "UV_PYTHON_INSTALL_DIR=/tmp/merlin_uvpy uv python install 3.13 && ls /tmp/merlin_uvpy && "
                        "UV_PYTHON_INSTALL_DIR=/tmp/merlin_uvpy UV_PYTHON_DOWNLOADS=never uv python find 3.13", timeout=900)
    if rc:
        fail("uv python install 3.13 failed")
    sh("py313_tar", f"tar -czf {PAY}/merlin_uvpy.tar.gz -C /tmp merlin_uvpy && du -sh {PAY}/merlin_uvpy.tar.gz")

    # 5. wheelhouse: every pinned package of the lock, cp313 manylinux (<= glibc 2.35), hash-checked by pip
    lock = T / "src/merlin-engine/release/requirements-0.32.lock"
    plats = " ".join(f"--platform manylinux_2_{m}_x86_64" for m in range(17, 36)) + \
        " --platform manylinux2014_x86_64 --platform manylinux1_x86_64 --platform linux_x86_64"
    rc, _ = sh("wheelhouse", f"python3 -m pip download --no-deps --only-binary=:all: --require-hashes --python-version 3.13 "
                             f"--implementation cp --abi cp313 --abi abi3 --abi none {plats} -r {lock} -d {PAY}/wheelhouse "
                             f"--progress-bar off -q", timeout=5400)
    pins = {re.sub(r"[-_.]+", "-", m.group(1)).lower() for m in re.finditer(r"^([A-Za-z0-9_.-]+)(?:\[[^\]]*\])?==", lock.read_text(), re.M)}
    have = {re.sub(r"[-_.]+", "-", p.name.split("-")[0]).lower() for p in (PAY / "wheelhouse").glob("*.whl")}
    R["wheelhouse"] = {"pins": len(pins), "wheels": len(list((PAY / "wheelhouse").glob("*.whl"))), "missing": sorted(pins - have),
                       "bytes": sum(p.stat().st_size for p in (PAY / "wheelhouse").glob("*"))}
    if rc or R["wheelhouse"]["missing"]:
        fail(f"wheelhouse: rc {rc}, missing {R['wheelhouse']['missing']}")

    # 6. the shim (system GCC 11; plain C, no version script)
    rc, _ = sh("shim", f"gcc -O2 -shared -fPIC -o {PAY}/libglibc238_shim.so {PAY}/glibc238_shim.c && "
                       f"nm -D --defined-only {PAY}/libglibc238_shim.so | grep isoc23")
    if rc:
        fail("shim build failed")

    # 7. diagnosis on the wheel's own .so files (before any install): what needs > 2.35, ldd before / after the fix
    D = T / "diag"
    shutil.rmtree(D, ignore_errors=True)
    D.mkdir()
    sh("diag_unzip", f"cd {D} && python3 -c \"import zipfile; z=zipfile.ZipFile('{PAY}/{WHEEL_SAFE}'); "
                     f"[z.extract(n) for n in z.namelist() if n.endswith('.so') or '.so.' in n.split('/')[-1]]\" && find . -name '*.so*' | head -40")
    sos = sorted(str(p) for p in D.rglob("*.so*") if p.is_file())
    rc, txt = sh("diag_scan", f"python3 -I {PAY}/glibc235_fix.py --dry-run " + " ".join(sos), ok_rc=(0, 2))
    m = re.search(r"^GLIBCFIX (.*)$", txt, re.M)
    R["diag_scan"] = json.loads(m.group(1)) if m else None
    if not m or rc:
        fail("glibc scan of the wheel .so files refused or failed (a symbol the shim does not cover?)")
    stub = T / "stub"
    stub.mkdir(exist_ok=True)
    if not (stub / "libcuda.so.1").exists():
        os.symlink("/usr/local/cuda-12.8/lib64/stubs/libcuda.so", stub / "libcuda.so.1")
    R["stub"] = str(stub)

    # 8. offline install from the payload only (the GPU notebook's exact path): fixed-prefix trees removed first
    shutil.rmtree("/tmp/merlin_gcc13", ignore_errors=True)
    shutil.rmtree("/tmp/merlin_uvpy", ignore_errors=True)
    shutil.rmtree("/tmp/merlin_off", ignore_errors=True)
    rc, txt = sh("offline_install", f"bash {PAY}/install_merlin_offline.sh {PAY} /tmp/merlin_off", timeout=5400)
    kv = dict(re.findall(r"^(MERLIN_[A-Z]+)=(.*)$", txt, re.M))
    R["offline_install"] = {"rc": rc, "ok": rc == 0 and "INSTALL OK" in txt, "vars": kv,
                            "glibcfix": [json.loads(x) for x in re.findall(r"^GLIBCFIX (.*)$", txt, re.M)],
                            "rehash": [json.loads(x) for x in re.findall(r"^REHASH (.*)$", txt, re.M)],
                            "verified": "ENVIRONMENT VERIFIED" in txt}
    if not R["offline_install"]["ok"] or not R["offline_install"]["verified"]:
        fail("offline install / --verify failed")

    # 9. load tests on the installed, patched tree: ldd (with / without shim) and the engine import
    if R["offline_install"]["ok"]:
        py, tree, shim = kv["MERLIN_VENV"] + "/bin/python", kv["MERLIN_TREE"], kv["MERLIN_SHIM"]
        rc, txt = sh("libpaths", f"{py} -c \"import torch, sysconfig, os; print(os.path.dirname(torch.__file__) + '/lib'); "
                                 f"print(sysconfig.get_paths()['purelib'] + '/nvidia/cu13/lib')\"")
        lp = ":".join(txt.strip().splitlines()[-2:] + [str(stub)])
        C = f"{tree}/vllm/_C_stable_libtorch.abi3.so"
        # v3: ldd prints "weak version `GLIBC_2.38' not found" for the weakened need (informational, ld.so does not fail on
        # a weak version); the failures are "undefined symbol" and "=> not found" (a missing library)
        sh("ldd_noshim", f"LD_LIBRARY_PATH={lp} ldd -r {C} 2>&1 | grep -E 'GLIBC|isoc23|not found' || echo CLEAN", ok_rc=(0, 1))
        rc, txt = sh("ldd_shim", f"LD_PRELOAD={shim} LD_LIBRARY_PATH={lp} ldd -r {C} 2>&1 | grep -E 'GLIBC|isoc23|not found' || echo CLEAN",
                     ok_rc=(0, 1))
        bad = [l for l in txt.splitlines() if "undefined symbol" in l or "=> not found" in l]
        R["ldd_shim_clean"] = not bad
        R["ldd_shim_lines"] = txt.strip().splitlines()[-10:]
        if not R["ldd_shim_clean"]:
            fail("ldd -r with the shim still reports GLIBC / isoc23 / not-found lines")
        imp = (f"import torch, vllm; import vllm._C_stable_libtorch as c; print('IMPORT OK', vllm.__version__, torch.__version__, "
               f"vllm.__file__)")
        sh("import_noshim", f"cd /tmp && LD_LIBRARY_PATH={stub} VLLM_USE_RUST_FRONTEND=0 {py} -c \"{imp}\"", ok_rc=(0, 1))
        rc, txt = sh("import_shim", f"cd /tmp && LD_PRELOAD={shim} LD_LIBRARY_PATH={stub} VLLM_USE_RUST_FRONTEND=0 {py} -c \"{imp}\"",
                     ok_rc=(0, 1))
        R["import_shim_ok"] = rc == 0 and "IMPORT OK" in txt
        if not R["import_shim_ok"]:
            fail("engine import with the shim failed")
        # launcher facts for the GPU notebook (profiles, dry-run commands); informational
        sh("profiles", f"ls {tree}/tools/merlin/profiles; ls {tree}/models/qwen38-27b", ok_rc=(0, 1, 2))
        for tp in (1, 2, 4):   # v3: merlin_cli serve --dry-run (launch.py has no --dry-run); simulated 4 x L4, no P2P
            sh(f"dryrun_tp{tp}", f"cd {tree} && {py} merlin_cli.py serve 27b --tp {tp} --dry-run --simulate 4xl4:nop2p",
               ok_rc=tuple(range(256)), timeout=300)
except Exception as e:
    import traceback
    fail(f"fatal: {e!r}")
    R["traceback"] = traceback.format_exc()[-3000:]
finally:
    shutil.rmtree(PAY / "__pycache__", ignore_errors=True)
    R["payload"] = {p.name: p.stat().st_size for p in PAY.iterdir() if p.is_file()}
    R["payload_bytes"] = sum(p.stat().st_size for p in PAY.rglob("*") if p.is_file())
    R["output_ok"] = R["payload_bytes"] < 19e9
    if not R["output_ok"]:
        fail(f"payload {R['payload_bytes'] / 1e9:.2f} GB too big")
    R["all_ok"] = not R["errors"]
    # v4 (Kaggle Model upload takes at most 50 files; the payload has 231): on ALL OK the payload becomes ONE stored zip
    # (wheels are already compressed), CRC-checked after writing, then the folder is removed; output = 3 files.
    # Kaggle unzips it on upload, giving merlin_offline/... inside the Model.
    import zipfile
    if R["all_ok"]:
        try:
            files = sorted(p for p in PAY.rglob("*") if p.is_file())
            odd = [str(p.relative_to(PAY)) for p in files if re.search(r"[^A-Za-z0-9._/-]", str(p.relative_to(PAY)))]
            R["odd_names"] = odd
            if odd:
                raise RuntimeError(f"file names Kaggle Models may refuse: {odd}")
            zpath = W / "merlin_offline.zip"
            with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED, allowZip64=True) as z:
                for p in files:
                    z.write(p, "merlin_offline/" + str(p.relative_to(PAY)))
            with zipfile.ZipFile(zpath) as z:
                bad_member = z.testzip()
                n_members = len(z.namelist())
            R["zip"] = {"file": zpath.name, "bytes": zpath.stat().st_size, "sha256": sha256(zpath), "members": n_members,
                        "files": len(files), "crc_bad_member": bad_member}
            if bad_member is not None or n_members != len(files):
                fail(f"zip check failed: bad member {bad_member}, {n_members} members vs {len(files)} files")
            else:
                shutil.rmtree(PAY)
        except Exception as e:
            fail(f"zip: {e!r}")
        R["all_ok"] = not R["errors"]
    R["sec"] = round(time.time() - T0)
    (W / "p2_report.json").write_text(json.dumps(R, indent=1))
    with zipfile.ZipFile(W / "p2_logs.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(LOGS.glob("*")):
            z.write(p, "p2_logs/" + p.name)
    R["output_files"] = sorted(p.name for p in W.iterdir())
    print("output files:", R["output_files"], json.dumps(R.get("zip")), flush=True)
    print(json.dumps({k: R.get(k) for k in ("merlin_commit", "wheel", "gcc13", "wheelhouse", "offline_install", "ldd_shim_clean",
                                             "import_shim_ok", "payload_bytes", "errors")}, default=str)[:6000], flush=True)
    print("ALL OK" if R["all_ok"] else "NOT OK", "|", R["sec"], "s", flush=True)
